# Part 1: Exploratory Data Analysis (EDA)
**Multi-Layer Phishing Detection & Risk Assessment**

This notebook provides visual and statistical analysis of the dataset, examining class distributions, URL structure, feature correlations, and security patterns.

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.eda import generate_eda_visualizations, generate_eda_report

## 1. Load Processed Feature Dataset

In [ ]:
features_csv = project_root / "dataset" / "processed" / "url_features.csv"
if not features_csv.exists():
    raise FileNotFoundError("url_features.csv not found. Please run main.py first.")

df = pd.read_csv(features_csv)
print(f"Dataset Shape: {df.shape}")
print(df.head())

## 2. Class Balance & Target Distribution

In [ ]:
counts = df['label'].value_counts()
print("Class Counts:")
print(f"  Legitimate (0): {counts[0]:,} ({counts[0] / len(df) * 100:.1f}%)")
print(f"  Phishing (1):   {counts[1]:,} ({counts[1] / len(df) * 100:.1f}%)")

fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df, x='label', palette={0: '#2ecc71', 1: '#e74c3c'}, ax=ax)
ax.set_xticklabels(['Legitimate (0)', 'Phishing (1)'])
ax.set_title("Target Class Distribution")
plt.show()

## 3. URL and Hostname Length Comparison

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df, x='label', y='url_length', palette={0: '#2ecc71', 1: '#e74c3c'}, ax=ax1, showfliers=False)
ax1.set_title("URL Length (Excl. Outliers)")
ax1.set_xticklabels(['Legitimate (0)', 'Phishing (1)'])

sns.boxplot(data=df, x='label', y='hostname_length', palette={0: '#2ecc71', 1: '#e74c3c'}, ax=ax2, showfliers=False)
ax2.set_title("Hostname Length (Excl. Outliers)")
ax2.set_xticklabels(['Legitimate (0)', 'Phishing (1)'])
plt.show()

## 4. Key Structural Features Comparison

In [ ]:
struct_cols = ['num_dots', 'num_hyphens', 'num_slashes', 'num_special_characters']
grouped_means = df.groupby('label')[struct_cols].mean()
print("Mean values by class:")
print(grouped_means)

grouped_means.T.plot(kind='bar', figsize=(10, 5), color=['#2ecc71', '#e74c3c'])
plt.title("Comparison of Mean Structural Counts by Class")
plt.legend(['Legitimate (0)', 'Phishing (1)'])
plt.xticks(rotation=0)
plt.show()

## 5. Suspicious-Pattern Indicator Prevalence

In [ ]:
patterns = ['has_suspicious_keyword', 'has_encoded_characters', 'has_ip_address', 'has_url_shortener']
pat_df = df.groupby('label')[patterns].mean() * 100
print("Prevalence (%):")
print(pat_df)

pat_df.T.plot(kind='bar', figsize=(10, 5), color=['#2ecc71', '#e74c3c'])
plt.title("Suspicious Pattern Prevalence (%) by Class")
plt.ylabel("Prevalence (%)")
plt.legend(['Legitimate (0)', 'Phishing (1)'])
plt.xticks(rotation=15)
plt.show()

## 6. Correlation Heatmap

In [ ]:
corr_cols = [
    'url_length', 'hostname_length', 'path_length', 'query_length',
    'num_dots', 'num_hyphens', 'num_slashes', 'num_digits',
    'has_ip_address', 'is_https', 'has_suspicious_keyword', 'label'
]
plt.figure(figsize=(10, 8))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title("Correlation Matrix with Target Label")
plt.show()